In [1]:
import os
import pandas as pd
from pathlib import Path
from transformers import T5ForConditionalGeneration, T5Tokenizer


c:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BASE_DIR  = Path(os.getcwd()).parent
INPUT_CSV = BASE_DIR / "data" / "processed" / "emails_with_sentiment.csv"

In [3]:
df = pd.read_csv(INPUT_CSV)
print(f"Loaded {len(df):,} emails")
df[['subject', 'body_clean', 'intent', 'sentiment']].head(3)

Loaded 8,064 emails


,subject,body_clean,intent,sentiment
0,Re:,Traveling to have a business meeting takes the...,meeting_request,NEGATIVE
1,NaN,"Randy,\n\n Can you send me a schedule of the s...",meeting_request,NEGATIVE
2,Re: Hello,"Greg,\n\n How about either next Tuesday or Thu...",greeting,NEGATIVE


In [11]:
print(" Loading Flan-T5 model")
model_name = "google/flan-t5-small"
tokenizer  = T5Tokenizer.from_pretrained(model_name)
model      = T5ForConditionalGeneration.from_pretrained(model_name)
print("Flan-T5 model loaded!")

 Loading Flan-T5 model


c:\Users\lenovo\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\lenovo\.cache\huggingface\hub\models--google--flan-t5-small. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 190/190 [00:00<00:00, 4312.86it/s]
[transformers] The tie

Flan-T5 model loaded!


In [19]:
def generate_reply(email_body, intent, sentiment):
    
    templates = {
        ("meeting_request", "POSITIVE"): [
            "Thank you for reaching out! I'd be happy to meet. Please let me know a convenient time and I'll confirm the appointment.",
            "Sure, I'm available to meet. Could you please share your preferred time slot so we can schedule accordingly?",
            "Great! I'd love to connect. Let's set up a meeting — please suggest a few time slots that work for you."
        ],
        ("meeting_request", "NEGATIVE"): [
            "Thank you for your message. I understand your concern and would like to discuss this further. Please let me know a suitable time to connect.",
            "I appreciate you reaching out. Let's schedule a call to address this — please share your availability.",
            "I'd like to resolve this matter at the earliest. Could we set up a meeting? Please suggest a convenient time."
        ],
        ("complaint", "NEGATIVE"): [
            "I sincerely apologize for the inconvenience caused. We are looking into this matter and will get back to you shortly.",
            "Thank you for bringing this to our attention. I'm sorry for the trouble and will ensure this is resolved immediately.",
            "We deeply regret the issue you've faced. Our team is working on it and you will hear from us within 24 hours."
        ],
        ("complaint", "POSITIVE"): [
            "Thank you for your feedback! We're glad you reached out and will make sure to address your concern promptly.",
            "We appreciate you letting us know. We'll look into this and get back to you as soon as possible.",
            "Thanks for sharing this with us. We'll take the necessary steps to ensure a better experience going forward."
        ],
        ("appreciation", "POSITIVE"): [
            "Thank you so much! It was a pleasure working with you and I truly appreciate your kind words.",
            "Many thanks! Your appreciation means a lot to us. We look forward to continuing to work together.",
            "Thank you for the kind feedback! We're glad we could help and hope to assist you again in the future."
        ],
        ("appreciation", "NEGATIVE"): [
            "Thank you for your message. We're sorry if we fell short of your expectations and will strive to do better.",
            "We appreciate your feedback and are committed to improving. Thank you for taking the time to share this.",
            "Thank you for reaching out. We take all feedback seriously and will work to improve our service."
        ],
        ("apology", "NEGATIVE"): [
            "Thank you for your understanding. I sincerely apologize for any inconvenience and assure you it won't happen again.",
            "I appreciate your patience. I'm sorry for the trouble caused and will make sure this is corrected immediately.",
            "Thank you for bringing this to my attention. I apologize for the oversight and will take corrective action right away."
        ],
        ("apology", "POSITIVE"): [
            "Thank you for being so understanding! I appreciate your patience and will ensure everything is sorted out.",
            "Your understanding means a lot. I'll make sure to address this promptly. Thank you!",
            "Thank you for your kind response. I'll take care of this right away and keep you updated."
        ],
        ("information_request", "POSITIVE"): [
            "Thank you for your inquiry! I'd be happy to provide the information you need. Please find the details below.",
            "Great question! Here's the information you requested. Please let me know if you need anything else.",
            "Thanks for reaching out. I'm happy to help — please find the requested details attached."
        ],
        ("information_request", "NEGATIVE"): [
            "Thank you for your message. I understand your concern and will provide the necessary information as soon as possible.",
            "I appreciate your patience. I'll look into this and share the relevant details with you shortly.",
            "Thank you for reaching out. I'll gather the required information and get back to you at the earliest."
        ],
        ("greeting", "POSITIVE"): [
            "Hello! Thank you for reaching out. Hope you're doing well too. How can I assist you today?",
            "Hi there! Great to hear from you. Please let me know how I can help.",
            "Hello! Thanks for your message. I'm doing well — hope the same for you! What can I do for you?"
        ],
        ("greeting", "NEGATIVE"): [
            "Hello, thank you for reaching out. I'm here to help — please let me know what you need.",
            "Hi, I received your message. I understand things might not be great right now — how can I assist?",
            "Hello! Thank you for getting in touch. Please share your concern and I'll do my best to help."
        ],
        ("farewell", "POSITIVE"): [
            "Thank you! It was great connecting with you. Looking forward to working together again soon!",
            "Thanks for everything! Wishing you all the best. Do reach out anytime you need assistance.",
            "It was a pleasure! Take care and hope to hear from you again soon."
        ],
        ("farewell", "NEGATIVE"): [
            "Thank you for your time. I hope we can resolve any pending matters soon. Take care.",
            "I appreciate your patience throughout. Wishing you well and hope things improve soon.",
            "Thank you for reaching out. I hope we can do better next time. Take care!"
        ],
        ("general", "POSITIVE"): [
            "Thank you for your message! I'll look into this and get back to you as soon as possible.",
            "Thanks for reaching out. I appreciate you sharing this and will respond promptly.",
            "Thank you! I've noted your message and will follow up shortly."
        ],
        ("general", "NEGATIVE"): [
            "Thank you for your message. I understand your concern and will address it as soon as possible.",
            "I appreciate you reaching out. I'll look into this matter and get back to you shortly.",
            "Thank you for letting me know. I'll make sure this is handled promptly."
        ],
    }

    # Intent + sentiment ke basis pe template lo
    key = (intent, sentiment)
    replies = templates.get(key, templates[("general", "POSITIVE")])

    return {
        "short":   replies[0],
        "formal":  replies[1],
        "detailed": replies[2]
    }

print("Template based reply function ready!")

Template based reply function ready!


In [20]:
print("Testing on 5 sample emails...\n")
print("=" * 60)

samples = df.sample(5, random_state=42)

for i, row in samples.iterrows():
    replies = generate_reply(row['body_clean'], row['intent'], row['sentiment'])
    print(f" Subject  : {row['subject']}")
    print(f"   Intent   : {row['intent']}")
    print(f"   Sentiment: {row['sentiment']}")
    print(f"\nShort   : {replies['short']}")
    print(f" Formal  : {replies['formal']}")
    print(f" Detailed: {replies['detailed']}")
    print("=" * 60)

Testing on 5 sample emails...

 Subject  : RE:
   Intent   : meeting_request
   Sentiment: NEGATIVE

Short   : Thank you for your message. I understand your concern and would like to discuss this further. Please let me know a suitable time to connect.
 Formal  : I appreciate you reaching out. Let's schedule a call to address this — please share your availability.
 Detailed: I'd like to resolve this matter at the earliest. Could we set up a meeting? Please suggest a convenient time.
 Subject  : astros tix
   Intent   : meeting_request
   Sentiment: POSITIVE

Short   : Thank you for reaching out! I'd be happy to meet. Please let me know a convenient time and I'll confirm the appointment.
 Formal  : Sure, I'm available to meet. Could you please share your preferred time slot so we can schedule accordingly?
 Detailed: Great! I'd love to connect. Let's set up a meeting — please suggest a few time slots that work for you.
 Subject  : FYI-Risk Moving
   Intent   : general
   Sentiment: NEGATI

In [21]:

df_sample = df.head(100).copy()
df_sample['generated_reply'] = df_sample.apply(
    lambda row: generate_reply(
        row['body_clean'],
        row['intent'],
        row['sentiment']
    ), axis=1
)
print("Done!")
print(f"Total replies generated: {len(df_sample):,}")

Done!
Total replies generated: 100


In [22]:
OUTPUT_CSV = BASE_DIR / "data" / "processed" / "emails_with_replies.csv"
df.to_csv(OUTPUT_CSV, index=False)
print(f" Saved → {OUTPUT_CSV}")

 Saved → d:\deep_learning_projects\Smart_email_reply\smart_email_reply\data\processed\emails_with_replies.csv
